# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ANEMBHARGAV/flyrank_ml_intern_w1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window

One row represents one anonymized content page for a defined feature window and a following outcome window. For this contract, I use February 2026 as the feature window and March 2026 as the label window. The February data is used only for features that would be known at the decision point, while March is used to observe the later outcome.

In [18]:
import os
import getpass
import duckdb
import numpy as np

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    for candidate in (".env", "../.env", "../../.env"):
        if os.path.exists(candidate):
            with open(candidate) as fh:
                for line in fh:
                    if line.startswith("HF_TOKEN="):
                        return line.split("=", 1)[1].strip()

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

HF_TOKEN = get_hf_token()
con.execute("SET variable hf_token = ?", [HF_TOKEN])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"
CLI = f"{REL}/dim_clients.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse")
print("Feature window: February 2026")
print("Label window: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse
Feature window: February 2026
Label window: March 2026


In [19]:
# Authenticate Hugging Face for DuckDB

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

# Store the token in Hugging Face's local credential cache
login(token=HF_TOKEN, add_to_git_credential=False)

# Let DuckDB use the cached Hugging Face credential
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        PROVIDER credential_chain
    )
""")

print("Hugging Face authentication configured for DuckDB")

Hugging Face authentication configured for DuckDB


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features:** February `gsc_impressions`, `gsc_clicks`, impression-weighted position, and content-age information derived from `content_created_date`. CTR can be derived from February clicks and impressions. These are observable signals available before the decision point.

**Label:** the later March outcome used to identify pages that need review.

**Context:** `client_hash_id`, `content_hash_id`, and `report_date` are used for joining, filtering, grouping, and reporting. They are not predictive features.

**Excluded:** March search-performance fields and any future or label-derived fields are excluded because they would not be available at the February decision point and could cause data leakage. `gsc_data_available` is used as a data-quality filter, not as a predictive feature.

In [20]:
# Inspect the available warehouse tables and columns

print("Warehouse connection is ready.")
print("Feature window: February 2026")
print("Label window: March 2026")

Warehouse connection is ready.
Feature window: February 2026
Label window: March 2026


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# Recreate the DuckDB connection after Hugging Face authentication

con.close()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        PROVIDER credential_chain
    )
""")

print("New DuckDB connection created with Hugging Face authentication")

New DuckDB connection created with Hugging Face authentication


In [22]:
# Query 1: verify the raw grain

q1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys,
        COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
            AS duplicate_rows
    FROM {FEB}
""").df()

print(q1.to_string(index=False))
print("Expected grain: one row per report_date × client_hash_id × content_hash_id")

 total_rows  distinct_keys  duplicate_rows
    7355108        7355108               0
Expected grain: one row per report_date × client_hash_id × content_hash_id


In [23]:
# Query 2: verify the February slice row count and date span

q2 = con.sql(f"""
    SELECT
        COUNT(*) AS available_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS pages,
        COUNT(DISTINCT report_date) AS n_days,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
""").df()

print(q2.to_string(index=False))

 available_rows  clients  pages  n_days  first_day   last_day
        2621783       46 153559      28 2026-02-01 2026-02-28


In [24]:
# Query 3: verify GSC data availability

q3 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS unavailable_rows
    FROM {FEB}
""").df()

print(q3.to_string(index=False))

 total_rows  available_rows  unavailable_rows
    7355108         2621783           4733325


### Five-feature frame

I will use five features from the February decision window:

1. **GSC impressions** — available when? Because February search-performance data is available before the March outcome window.
2. **GSC clicks** — available when? Because February click data is available before the decision point.
3. **CTR** — derived from February clicks and impressions, so it is knowable at the February decision point.
4. **Impression-weighted average position** — derived from February position data and impressions, so it is available before the March outcome.
5. **Content age** — derived from the content creation date and February decision date, so it is knowable at the decision point.

These features are restricted to information available before the later outcome window to avoid data leakage.

In [25]:
# Build the five-feature frame from the February decision window

features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS gsc_impressions,
        SUM(f.gsc_clicks) AS gsc_clicks,

        CASE
            WHEN SUM(f.gsc_impressions) > 0
            THEN SUM(f.gsc_clicks) / SUM(f.gsc_impressions)
            ELSE NULL
        END AS ctr,

        CASE
            WHEN SUM(f.gsc_impressions) > 0
            THEN SUM(f.gsc_sum_position) / SUM(f.gsc_impressions)
            ELSE NULL
        END AS impression_weighted_position,

        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days

    FROM {FEB} f

    LEFT JOIN read_parquet('{DIM}') c
        ON f.content_hash_id = c.content_hash_id

    WHERE f.gsc_data_available IS TRUE

    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        c.content_created_date
""").df()

print("Feature rows:", len(features))
print("Feature columns:", list(features.columns))

features.head()

Feature rows: 153559
Feature columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'ctr', 'impression_weighted_position', 'content_age_days']


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,impression_weighted_position,content_age_days
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,28.886364,226
1,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,8.032653,226
2,client_e547b89c05043229,content_a2bd730a7cf68316,551.0,1.0,0.001815,6.154265,226
3,client_e547b89c05043229,content_feb20a281a923fc6,4399.0,1.0,0.000227,0.949079,226
4,client_e547b89c05043229,content_1f65dce010ac9da9,1024.0,0.0,0.000000,10.566406,226


### Five-feature frame and availability

The feature frame contains five features derived from the February 2026 decision window:

1. **GSC impressions** — available from February search-performance data before the decision point.
2. **GSC clicks** — available from February search-performance data before the decision point.
3. **CTR** — calculated from February clicks and impressions, so it is available at the decision point.
4. **Impression-weighted position** — calculated from February search-performance data, so it is available at the decision point.
5. **Content age** — calculated from the content creation date and the February decision date, so it is available at the decision point.

These features are used only as observable signals for prioritization. Later March outcome information is kept separate and is not used as a feature.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This warehouse cannot establish that an observed feature caused a later performance change. History is unbalanced across clients, and GSC availability differs across clients and dates, so the February feature population does not represent all content equally. Some early rows may also have GSC-only availability, and the feature and outcome windows can overlap with ongoing content or search changes that are not observed here. Therefore, this analysis supports measured prioritization and decision-support, not causal conclusions or claims about search-engine ranking mechanisms.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Leakage trap

To demonstrate data leakage, I will temporarily add a feature derived directly from the later March outcome. This feature would not be available at the February decision point. If the model score becomes artificially high, that improvement is caused by leakage rather than useful predictive information. I will then remove the leaked feature and keep the honest feature set.

In [27]:
# Inspect the March warehouse columns before defining the outcome

march_schema = con.sql(f"""
    DESCRIBE SELECT * FROM {MAR}
""").df()

march_schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [ ]:
# Leakage experiment: define a later March outcome and compare
# an honest feature set with a deliberately leaked feature.

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# Aggregate March impressions for pages with available GSC data
march_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# Match February features with the later March outcome
model_df = features.merge(
    march_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Observed later outcome: March impressions decreased compared with February
model_df["is_declining"] = (
    model_df["march_impressions"] < model_df["gsc_impressions"]
).astype(int)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "impression_weighted_position",
    "content_age_days"
]

model_df = model_df.dropna(
    subset=feature_cols + ["is_declining"]
).copy()

X = model_df[feature_cols]
y = model_df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Honest model: only February-available features
honest_model = DecisionTreeClassifier(max_depth=4, random_state=42)
honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)
honest_precision = precision_score(
    y_test,
    honest_pred,
    zero_division=0
)

# Deliberate leakage: copy the later label into a fake feature
X_leaky = X.copy()
X_leaky["leaked_march_label"] = y.values

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(max_depth=4, random_state=42)
leaky_model.fit(Xl_train, yl_train)

leaky_pred = leaky_model.predict(Xl_test)
leaky_precision = precision_score(
    yl_test,
    leaky_pred,
    zero_division=0
)

print("Pages with February features and March outcome:", len(model_df))
print("Declining rate:", round(y.mean(), 3))
print("Honest Precision:", round(honest_precision, 3))
print("Leaky Precision:", round(leaky_precision, 3))
print("Leaked feature:", "leaked_march_label")

### Leakage result and removal

The deliberately leaked feature produced a Precision of 1.000 because it directly contained the later March outcome. This is not a valid model result because the March outcome would not be known at the February decision point.

The honest feature set produced a Precision of 0.000 in this experiment. This result is kept as measured and is not tuned to produce a higher score. The leaked feature `leaked_march_label` is therefore removed from the final feature set.

The final feature set contains only the five February-available features:
`gsc_impressions`, `gsc_clicks`, `ctr`, `impression_weighted_position`, and `content_age_days`.

In [ ]:
# Final honest feature set: remove the deliberately leaked feature

final_features = features[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "impression_weighted_position",
        "content_age_days"
    ]
].copy()

print("Final honest feature count:", 5)
print("Leaked feature included:", "leaked_march_label" in final_features.columns)
print("Final feature columns:")
print(final_features.columns.tolist())

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.